# Hermes Enhanced Misprice — Interactive Backtest

Run the synthetic universe, print metrics, and peek at equity / calibration.
Requires `PYTHONPATH` at the repo root.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / 'backtest').is_dir() is False and (ROOT.parent / 'backtest').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from backtest.engine import BacktestEngine
from backtest.metrics import compute_metrics, calibration_points, threshold_sweep
from backtest.plots import save_equity_and_drawdown, save_calibration
from models.config import load_enhanced_config
from IPython.display import display, Markdown

cfg = load_enhanced_config()
cfg.synthetic_n_markets = 4000
er = BacktestEngine(cfg, mode='enhanced', seed=42).run_synthetic(n_markets=4000, seed=42)
m = compute_metrics(er)
display(Markdown(f"**Win rate:** {m.win_rate:.1%} · **Max DD:** {m.max_drawdown_pct:.1%} · **Brier:** {m.brier:.3f}"))
print(m.plain_english)
out = Path('artifacts/backtest_runs/notebook_preview')
out.mkdir(parents=True, exist_ok=True)
save_equity_and_drawdown(er.equity_curve, out / 'equity.png')
save_calibration(calibration_points(er.decisions), out / 'calibration.png')
print('Saved plots to', out)
threshold_sweep(er.decisions)[:5]